In [2]:
"""
Download NYISO hourly zonal actual load (palIntegrated dataset).
Source: mis.nyiso.com/public - Integrated Real-Time Actual Load.
Output: one clean parquet, 11 zone columns + statewide sum, UTC index.

Idempotent: cached ZIPs are skipped on re-run. Safe after pod restarts.
"""

import io
import time
import zipfile
from pathlib import Path

import pandas as pd
import requests

# ----------------------------- configuration -----------------------------
START = "2013-01"
END = "2026-05"
BASE_URL = "http://mis.nyiso.com/public/csv/palIntegrated"
CACHE_DIR = Path("nyiso_zonal_raw")
OUT_PATH = Path("nyiso_zonal_hourly.parquet")

ZONES = {
    "WEST": "A", "GENESE": "B", "CENTRL": "C", "NORTH": "D",
    "MHK VL": "E", "CAPITL": "F", "HUD VL": "G", "MILLWD": "H",
    "DUNWOD": "I", "N.Y.C.": "J", "LONGIL": "K",
}

# --------------------------- step 1: download ----------------------------
def download_month(yyyymm: str) -> Path | None:
    fname = f"{yyyymm}01palIntegrated_csv.zip"
    local = CACHE_DIR / fname
    if local.exists() and local.stat().st_size > 0:
        return local
    url = f"{BASE_URL}/{fname}"
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200:
                local.write_bytes(r.content)
                return local
            print(f"  {yyyymm}: HTTP {r.status_code} - no archive at {url}")
            return None
        except requests.RequestException as e:
            print(f"  {yyyymm}: attempt {attempt + 1} failed ({e}), retrying...")
            time.sleep(5)
    return None


def month_range(start: str, end: str) -> list[str]:
    dates = pd.period_range(start=start, end=end, freq="M")
    return [p.strftime("%Y%m") for p in dates]


# ---------------------------- step 2: parse ------------------------------
def parse_zip(zip_path: Path) -> pd.DataFrame:
    frames = []
    with zipfile.ZipFile(zip_path) as zf:
        for name in sorted(zf.namelist()):
            if not name.endswith(".csv"):
                continue
            with zf.open(name) as f:
                df = pd.read_csv(io.TextIOWrapper(f, encoding="utf-8"))
            frames.append(df)
    return pd.concat(frames, ignore_index=True)


def find_load_column(df: pd.DataFrame) -> str:
    """
    Auto-detect the load value column. The hourly integrated dataset labels it
    'Integrated Load'; other NYISO datasets use 'Load'. Fall back to any single
    column containing 'load' (case-insensitive) so a future rename can't
    silently break us.
    """
    for exact in ("Integrated Load", "Load"):
        if exact in df.columns:
            return exact
    candidates = [c for c in df.columns if "load" in c.lower()]
    if len(candidates) == 1:
        return candidates[0]
    raise ValueError(
        f"Could not identify load column. Columns found: {list(df.columns)}"
    )


def to_utc(df: pd.DataFrame) -> pd.DataFrame:
    """EST = UTC-5, EDT = UTC-4 via the explicit Time Zone column."""
    ts = pd.to_datetime(df["Time Stamp"], format="%m/%d/%Y %H:%M:%S")
    offset = df["Time Zone"].map({"EST": 5, "EDT": 4})
    if offset.isna().any():
        bad = df.loc[offset.isna(), "Time Zone"].unique()
        raise ValueError(f"Unexpected Time Zone values: {bad}")
    df = df.copy()
    df["utc"] = ts + pd.to_timedelta(offset, unit="h")
    return df


# ----------------------------- main pipeline -----------------------------
def main() -> None:
    CACHE_DIR.mkdir(exist_ok=True)
    months = month_range(START, END)
    print(f"Downloading {len(months)} monthly archives "
          f"({months[0]} to {months[-1]})...")

    all_frames, failed = [], []
    for i, m in enumerate(months, 1):
        zp = download_month(m)
        if zp is None:
            failed.append(m)
            continue
        all_frames.append(parse_zip(zp))
        if i % 12 == 0:
            print(f"  ...{i}/{len(months)} months done")

    if failed:
        print(f"\nWARNING - {len(failed)} months failed: {failed}")

    print("\nParsing and pivoting...")
    long_df = pd.concat(all_frames, ignore_index=True)

    # show the actual schema once, so there is no more guessing
    print(f"Columns found in raw files: {list(long_df.columns)}")
    load_col = find_load_column(long_df)
    print(f"Using '{load_col}' as the load value column.")

    long_df = to_utc(long_df)

    found = set(long_df["Name"].unique())
    expected = set(ZONES.keys())
    if found != expected:
        print(f"NOTE - zone name mismatch.\n  missing: {expected - found}"
              f"\n  unexpected: {found - expected}")

    long_df = long_df.drop_duplicates(subset=["utc", "Name"], keep="last")

    wide = long_df.pivot(index="utc", columns="Name", values=load_col)
    wide = wide[[z for z in ZONES if z in wide.columns]]
    wide["NYISO_TOTAL"] = wide.sum(axis=1)
    wide = wide.sort_index()

    # ------------------------- diagnostics report -------------------------
    print("\n================ DIAGNOSTICS ================")
    print(f"Rows (hours):        {len(wide):,}")
    print(f"Range (UTC):         {wide.index.min()}  ->  {wide.index.max()}")
    full_range = pd.date_range(wide.index.min(), wide.index.max(), freq="h")
    missing_hours = full_range.difference(wide.index)
    print(f"Missing hours:       {len(missing_hours)}")
    if len(missing_hours) > 0:
        print(f"  first few: {missing_hours[:5].tolist()}")
    print(f"NaN cells per zone:\n{wide.isna().sum().to_string()}")
    print(f"\nStatewide total - min: {wide['NYISO_TOTAL'].min():,.0f} MW, "
          f"max: {wide['NYISO_TOTAL'].max():,.0f} MW, "
          f"mean: {wide['NYISO_TOTAL'].mean():,.0f} MW")
    print("=============================================")

    wide.to_parquet(OUT_PATH)
    print(f"\nSaved {OUT_PATH} ({OUT_PATH.stat().st_size / 1e6:.1f} MB)")


if __name__ == "__main__":
    main()

  ...12/161 months done


  ...24/161 months done


  ...36/161 months done


  ...48/161 months done


  ...60/161 months done


...

  ...144/161 months done


  ...156/161 months done

Parsing and pivoting...


Columns found in raw files: ['Time Stamp', 'Time Zone', 'Name', 'PTID', 'Integrated Load']
Using 'Integrated Load' as the load value column.



================ DIAGNOSTICS ================
Rows (hours):        117,575
Range (UTC):         2013-01-01 05:00:00  ->  2026-06-01 03:00:00
Missing hours:       0
NaN cells per zone:
Name
WEST           1
GENESE         1
CENTRL         1
NORTH          1
MHK VL         1
CAPITL         1
HUD VL         1
MILLWD         1
DUNWOD         1
N.Y.C.         1
LONGIL         1
NYISO_TOTAL    0

Statewide total - min: 0 MW, max: 33,956 MW, mean: 17,727 MW



Saved nyiso_zonal_hourly.parquet (8.8 MB)


In [3]:
import pandas as pd

df = pd.read_parquet("nyiso_zonal_hourly.parquet")
zones = [c for c in df.columns if c != "NYISO_TOTAL"]

# locate the bad hour
bad = df[df[zones].isna().any(axis=1)]
print("Missing hour(s):")
print(bad)

# interpolate zone-by-zone, recompute the total honestly
df[zones] = df[zones].interpolate(method="linear", limit=2)
df["NYISO_TOTAL"] = df[zones].sum(axis=1)

# verify
assert df[zones].isna().sum().sum() == 0, "still have NaNs"
print(f"\nAfter fix - min: {df['NYISO_TOTAL'].min():,.0f} MW "
      f"(should now be a plausible overnight low, ~11,000-13,000)")

df.to_parquet("nyiso_zonal_hourly.parquet")
print("Saved.")

Missing hour(s):
Name                 WEST  GENESE  CENTRL  NORTH  MHK VL  CAPITL  HUD VL  \
utc                                                                        
2016-01-29 03:00:00   NaN     NaN     NaN    NaN     NaN     NaN     NaN   

Name                 MILLWD  DUNWOD  N.Y.C.  LONGIL  NYISO_TOTAL  
utc                                                               
2016-01-29 03:00:00     NaN     NaN     NaN     NaN          0.0  

After fix - min: 10,731 MW (should now be a plausible overnight low, ~11,000-13,000)


Saved.


In [5]:
import pandas as pd

demand = pd.read_csv("data/processed/tft_clean.csv")
print(demand.columns.tolist())
print(demand.head(3))
print(demand.tail(3))
print(f"\nRows: {len(demand):,}")

['ts', 'demand', 'net_generation', 'total_interchange', 'hour', 'day_of_week', 'day_of_month', 'month', 'year', 'is_holiday', 'time_idx', 'series', 'split', 'act_temperature_2m', 'act_relative_humidity_2m', 'act_dew_point_2m', 'act_apparent_temperature', 'act_precipitation', 'act_snowfall', 'act_cloud_cover', 'act_surface_pressure', 'act_wind_speed_10m', 'act_wind_gusts_10m', 'act_shortwave_radiation', 'act_direct_radiation', 'act_diffuse_radiation', 'fc_temperature_2m', 'fc_relative_humidity_2m', 'fc_dew_point_2m', 'fc_apparent_temperature', 'fc_precipitation', 'fc_snowfall', 'fc_cloud_cover', 'fc_surface_pressure', 'fc_wind_speed_10m', 'fc_wind_gusts_10m', 'fc_shortwave_radiation', 'fc_direct_radiation', 'fc_diffuse_radiation', 'fc_is_synthetic', 'fx_app_roll72', 'fx_cdh_24h', 'fx_hot_streak_day', 'fx_night_min_app_prev']
                    ts   demand  net_generation  total_interchange  hour  \
0  2015-07-01 01:00:00  16891.0         14444.0            -2447.0     1   
1  2015-07-0